# Phase 4 — Central Match Logic Agent

This notebook performs centralized profile-based match decisions.

Scope:
- Subscribe to candidate events from bar agents
- Apply profile matching rules in order
- Evaluate probability once per newly eligible pair
- Enforce first-valid-candidate tie behavior
- Publish definitive match events

In [1]:
from datetime import datetime, timezone
import json
from pathlib import Path
import random
import sys
import uuid

cwd = Path.cwd()
candidate_paths = [cwd / "src", cwd.parent / "src"]
for path in candidate_paths:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher
import yaml

In [ ]:
if "connector" in globals() and connector is not None:
    try:
        connector.disconnect()
    except Exception:
        pass

cfg = load_config()

config_path = Path("config.yaml")
if not config_path.exists():
    config_path = Path("../config.yaml")

raw_cfg = {}
if config_path.exists():
    raw_cfg = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}
sim_cfg = raw_cfg.get("dating_simulation", {})

HEIGHT_TOLERANCE_CM = int(sim_cfg.get("height_tolerance_cm", 10))
HAIR_DISTANCE_EXACT = int(sim_cfg.get("hair_distance_exact", 10))
MATCH_PROB_SAME_HAIR = float(sim_cfg.get("match_prob_same_hair", 1/3))
MATCH_PROB_DISTANCE_10 = float(sim_cfg.get("match_prob_distance_10", 2/3))
MATCH_RANDOM_SEED = int(sim_cfg.get("match_random_seed", 42))

random.seed(MATCH_RANDOM_SEED)

base_topic = cfg.mqtt.base_topic
topic_candidate = f"{base_topic}/events/candidate"
topic_match = f"{base_topic}/events/match"

matched_people = set()
processed_pairs = set()
published_matches = 0

connector = MqttConnector(cfg.mqtt, client_id_suffix="agent-match-logic")
connector.connect()
is_connected = connector.wait_for_connection(timeout=3.0)
publisher = MqttPublisher(connector) if is_connected else None

def maybe_publish_match(payload: dict):
    global published_matches

    bar_id = payload.get("bar_id")
    if not isinstance(bar_id, str) or not bar_id:
        return

    try:
        person_a_id = int(payload.get("person_a_id"))
        person_b_id = int(payload.get("person_b_id"))
        height_a = int(payload.get("height_cm_a"))
        height_b = int(payload.get("height_cm_b"))
        hair_a = int(payload.get("hair_color_index_a"))
        hair_b = int(payload.get("hair_color_index_b"))
    except (TypeError, ValueError):
        return

    if person_a_id == person_b_id:
        return

    pair = (bar_id, min(person_a_id, person_b_id), max(person_a_id, person_b_id))
    person_a_key = (bar_id, person_a_id)
    person_b_key = (bar_id, person_b_id)

    if pair in processed_pairs:
        return
    if person_a_key in matched_people or person_b_key in matched_people:
        return

    if abs(height_a - height_b) > HEIGHT_TOLERANCE_CM:
        processed_pairs.add(pair)
        return

    hair_distance = abs(hair_a - hair_b)
    if hair_distance == 0:
        probability = MATCH_PROB_SAME_HAIR
    elif hair_distance == HAIR_DISTANCE_EXACT:
        probability = MATCH_PROB_DISTANCE_10
    else:
        processed_pairs.add(pair)
        return

    processed_pairs.add(pair)
    if random.random() > probability:
        return

    matched_people.add(person_a_key)
    matched_people.add(person_b_key)
    published_matches += 1

    match_event = {
        "match_id": str(uuid.uuid4()),
        "person_a_id": person_a_id,
        "person_b_id": person_b_id,
        "bar_id": bar_id,
        "height_cm_a": height_a,
        "height_cm_b": height_b,
        "hair_color_index_a": hair_a,
        "hair_color_index_b": hair_b,
        "duration_seconds": payload.get("duration_seconds", 0),
        "confirmed_by": "agent_match_logic",
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

    if publisher is not None:
        publisher.publish_json(topic_match, json.dumps(match_event), qos=0, retain=False)
    print(f"[match-published] #{published_matches} pair={pair} bar={bar_id}")

if not is_connected:
    print(f"Could not confirm MQTT connection to {cfg.mqtt.host}:{cfg.mqtt.port}")
else:
    def on_message(client, userdata, message):
        try:
            payload = json.loads(message.payload.decode("utf-8"))
        except Exception:
            return

        if message.topic == topic_candidate:
            maybe_publish_match(payload)

    connector.client.on_message = on_message
    connector.client.subscribe(topic_candidate)
    print(f"Connected to {cfg.mqtt.host}:{cfg.mqtt.port}")
    print(f"Subscribed to: {topic_candidate}")
    print(f"Publishing match events to: {topic_match}")

Connected to 127.0.0.1:1883
Subscribed to: simulated-city/events/candidate
Publishing match events to: simulated-city/events/match


[match-published] #1 pair=('bar_1', 1, 39) bar=bar_1
[match-published] #2 pair=('bar_1', 18, 34) bar=bar_1


In [ ]:
# Run this cell when finished
if "connector" in globals() and connector is not None:
    connector.disconnect()
    print("Match logic disconnected.")